In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Detecting Errors in Quantum Computations

This lab is designed as an exploratory example of using detecting errors in quantum simulations.  While a simple surface code example would be ideal, even the smallest surface would be too expensive to simulate in these notebooks with reasonably interactive run times. In this lab, we will explore a small component that enters many error correction schemes, dual rail qubit error detection.

We will run a circuit that is designed to create what is known as a Bell state, this is the equal superposition the $\ket{0}$ and $\ket{1}$ state: $$\ket{\textrm{bell}} = \frac{\ket{0} + \ket{1}}{\sqrt{2}}$$

We will run a circuit that create this state on two qubits as well as on a set of four qubits that are being treated as two dual-rail qubits.


In a dual rail qubit, we use two qubits to represent one qubit with two valid states.
\begin{equation*}
    \ket{0}_{\textrm{dual-rail}} = \ket{10}
\end{equation*}

\begin{equation*}
    \ket{1}_{\textrm{dual-rail}} = \ket{01}
\end{equation*}

Now if we ever find this pair in either state of $\ket{00}$ and $\ket{11}$ we know an error has occurred.  In most modern types of qubits, errors of the form $\ket{1}\rightarrow\ket{0}$ are the most common, which is why the idea of dual rail qubits is such a powerful tool.  This class of errors are known depolarizing errors, and dual-rail qubits can detect most depolarizing errors.  


We will run series of simulations of the bell state preparation circuit both on a normal pair of qubits, and on a set of four qubits configured into two coupled dual-rail qubits.

These simulations will be for a series of exaggerated depolarizing errors rates to that you can see the impact of detection.

# First, we need some helper functions.  

Please run the long cell below.  Qiskit does not have a native way to represnt dual-rail qubits and operations, so we wrote some helper functions to abstract way the underlying Qiskit mess.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

import numpy as np

# ══════════════════════════════════════════════════════════════════════
# Dual-rail gate helpers
# ══════════════════════════════════════════════════════════════════════

def prepare_logical_0(qc, rail0, rail1):
    """Prepare |0⟩_L = |10⟩"""
    qc.x(rail0)

def prepare_logical_1(qc, rail0, rail1):
    """Prepare |1⟩_L = |01⟩"""
    qc.x(rail1)

def logical_x(qc, rail0, rail1):
    """Logical X"""
    qc.swap(rail0, rail1)

def logical_hadamard(qc, rail0, rail1):
    """Logical Hadamard"""
    qc.cx(rail0, rail1)
    qc.ry(-np.pi / 2, rail0)
    qc.cx(rail0, rail1)

def logical_cnot(qc, ctrl_r0, ctrl_r1, targ_r0, targ_r1):
    """Logical CNOT via Fredkin (controlled-SWAP) gate"""
    qc.cswap(ctrl_r1, targ_r0, targ_r1)



# ══════════════════════════════════════════════════════════════════════
# Noise model builder
# ══════════════════════════════════════════════════════════════════════

def build_noise_model(p1: float, p2: float):
    """
    Build a depolarizing noise model.

    Args:
        p1: single-qubit gate depolarizing probability  (0 → 1)
        p2: two-qubit gate depolarizing probability     (0 → 1)

    The depolarizing channel applies I, X, Y, Z each with prob p/4
    (plus I with prob 1-p), uniformly randomising the qubit state.
    For dual-rail qubits this is informative because:
      - X errors on one rail → |10⟩ ↔ |01⟩  (possible undetected logical flip)
      - X errors on BOTH rails simultaneously → |10⟩ → |01⟩ or vice versa
      - Z errors → phase errors (undetected in computational basis measurements)
      - Y errors → combine bit and phase
      - Errors that produce |00⟩ or |11⟩ are DETECTED by codespace check
    """
    noise_model = NoiseModel()

    if p1 > 0:
        err1 = depolarizing_error(p1, 1)
        noise_model.add_all_qubit_quantum_error(err1, ['h', 'x', 'rx', 'ry', 'rz', 'u'])

    if p2 > 0:
        err2 = depolarizing_error(p2, 2)
        noise_model.add_all_qubit_quantum_error(err2, ['cx', 'swap'])
        # cswap is a 3-qubit gate and needs a 3-qubit error channel
        err3 = depolarizing_error(p2, 3)
        noise_model.add_all_qubit_quantum_error(err3, ['cswap'])

    return noise_model


# ══════════════════════════════════════════════════════════════════════
# Circuit builder
# ══════════════════════════════════════════════════════════════════════

def build_dual_rail_bell_circuit():
    """
    Build the logical Bell-state circuit:
      Start: |0⟩_A |0⟩_B
      Apply: Hadamard_A  then  CNOT(A→B)
      Expected (noiseless): (|0⟩_A|0⟩_B + |1⟩_A|1⟩_B) / √2
                          = (|10⟩|10⟩ + |01⟩|01⟩) / √2
    """
    qr = QuantumRegister(4, "q")
    cr = ClassicalRegister(4, "c")
    qc = QuantumCircuit(qr, cr)

    qc.barrier(label="Prepare")
    prepare_logical_0(qc, qr[0], qr[1])   # A = |0⟩_L
    prepare_logical_0(qc, qr[2], qr[3])   # B = |0⟩_L

    qc.barrier(label="Had_A")
    logical_hadamard(qc, qr[0], qr[1])

    qc.barrier(label="CNOT")
    logical_cnot(qc, qr[0], qr[1], qr[2], qr[3])

    qc.barrier(label="Measure")
    qc.measure(qr, cr)
    return qc

def build_simple_bell_circuit():
    """Simple state-prep only circuit, easier to reason about."""
    qr = QuantumRegister(2, "q")
    cr = ClassicalRegister(2, "c")
    qc = QuantumCircuit(qr, cr)
    qc.barrier(label="Had_A")
    qc.h(1)
    qc.barrier(label="CNOT")
    qc.cx(1, 0)
    qc.barrier(label="Measure")
    qc.measure(qr, cr)
    return qc

# ══════════════════════════════════════════════════════════════════════
# Single experiment runner
# ══════════════════════════════════════════════════════════════════════

def run_noisy_experiment(qc, p_err=0.,
                         shots=4096, show_circuit=False):
    # if show_circuit:
    #     print(f"\n{qc.draw(output='text', fold=80)}")
    
    noise_model = build_noise_model(p1=p_err, p2=2 * p_err)
    simulator = AerSimulator(noise_model=noise_model)
    tqc = transpile(qc, simulator)
    result = simulator.run(tqc, shots=shots).result()
    counts = result.get_counts()
    return counts

def run_noisy_dual_rail_bell_circuit(p_err):
    return run_noisy_experiment(build_dual_rail_bell_circuit(), p_err)

def run_noisy_simple_bell_circuit(p_err):
    return run_noisy_experiment(build_simple_bell_circuit(), p_err)

def split_valid_from_invalid(counts):
    valid = {}
    invalid = counts.copy()
    for bitstring in ('0101', '1010', '0110', '1001'):
        if counts.get(bitstring, None):
            valid[bitstring] = invalid.pop(bitstring)
    return valid, invalid

def summarize_valid_invalid(valid_invalid):
    valid,invalid = valid_invalid
    total = sum(list(valid.values()) + list(invalid.values()))  
    valid_total = sum(list(valid.values()))
    invalid_total = sum(list(invalid.values()))
    total = valid_total + invalid_total
    correct = valid.get('0101', 0) + valid.get('1010', 0)
    p_vec = np.array([valid.get('0101', 0), valid.get('1010', 0), valid.get('0110', 0), valid.get('1001', 0) ]) / valid_total
    p_vec -= np.array([.5, .5, 0, 0])
    infidelity = np.sqrt(p_vec @ p_vec)

    p_vec = np.array([valid.get('0101', 0), valid.get('1010', 0), valid.get('0110', 0), valid.get('1001', 0) ]) / total
    p_vec -= np.array([.5, .5, 0, 0])
    infidelity_t = np.sqrt(p_vec @ p_vec + invalid_total**2 / total**2)


    print(f'Detected Error Rate: {invalid_total / total * 100:.2f}%')
    print(f'Undetected Error Rate: {((valid_total - correct) / total) * 100:.2f}%')
    print(f'Potentially Correct Rate: {(correct / total) * 100:.2f}%')
    print(f'Filtered Correct Rate: {(correct / valid_total) * 100:.2f}%')
    print(f'Fidelity (valid space): {100 - (infidelity) * 100:.2f}%')
    print(f'Fidelity (total space): {100 - (infidelity_t) * 100:.2f}%')


**run_noisy_dual_rail_bell_circuit** 

and 

**run_noisy_simple_bell_circuit** 

will run dual-rail and normal Bell circuits.  The only parameter needed is the error rate.  Current modern qubits are around .0001 to .001, however you should explore to see what happens as you look at higher errors, such as .02 or .2?  The impact and utility of the error detection becomes more apparent.

In [ ]:
run_noisy_dual_rail_bell_circuit(p_err=0)

How do you expect the outputs to change if you were to increase p_err from 0 to .01?

In this zero error limit, why that the two possible states not identically equal?

**build_dual_rail_bell_circuit** will build the dual rail bell circuit using 4 qubits, while **build_simple_bell_circuit** will build the normal qubit version.  

Use circuit.draw() to see how the complexity increases between the two forms?

In [ ]:
build_simple_bell_circuit().draw(output='mpl')

**split_valid_from_invalid** 

and 

**summarize_valid_invalid** can help give some approximate summary information.  They essentially tally the simulated runs into valid possible final states, combinations of '01' and '10' and invalid final states, any combination with a '00' or '11'.



In [ ]:
summarize_valid_invalid(split_valid_from_invalid(run_noisy_dual_rail_bell_circuit(p_err=0)))

How does the fidelity computed from the valid shots compare to that using all shots?  Why?